# Teach RNN to see TAM

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# System imports
from pathlib import Path
import sys
from collections import defaultdict
import json

# Plotting imports
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# Scientific imports
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from scipy.stats import zscore, spearmanr
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram

# NN imports
import torch
import neurogym as ngym

# For import from src folder in linux
ROOT = Path().cwd().parent
SRC_PATH = ROOT / 'src'
sys.path.append(str(ROOT))
sys.path.append(str(SRC_PATH))

# Custom imports
from src.environments import TAMTask, MotionTask, load_motion_from_image, load_tam_from_image
from src.train_test_utils import plot_dataset, plot_training_curves
from src.networks import TAMNet
from src.utils import load_params

In [4]:
# Set directories
# Network
MODEL_DIR = ROOT / 'data' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Stimuli
STIM_DIR = ROOT / 'data' / 'stimuli' / "motion_task"

In [186]:
# Load parameters
params = load_params(str(ROOT / 'parameters.json'))
exp_params = params["EXPERIMENT"]
env_params = params['TASK']
net_params = params['NETWORK']

motion_dirs = ["horizontal"]
box_shapes = ["square", "circle", "triangle"]

img_size = 32
batch_size = 128
seq_len = 100

In [187]:
training_envs = defaultdict()
training_datasets = defaultdict()

# Create environments and datasets
for md in motion_dirs:
    for bs in box_shapes:
        print(md, bs)
        stimuli = load_motion_from_image(img_size, stim_path=STIM_DIR)
        env = MotionTask(
            box_shape=bs,
            # stim_type="tam",
            motion_type="cnt",
            stimuli=stimuli,
            stim_ori=md,
            dt=env_params["dt"],
            # sigma=env_params["sigma"],
            sigma=0,
            img_size=img_size
        )
        env.reset(no_step=True)

        # Generate dataset object
        dataset = ngym.Dataset(
            env,
            batch_size=batch_size,
            seq_len=seq_len
        )

        # Add dataset to dictionary
        training_envs[f'{bs}-{md}'] = env
        training_datasets[f'{bs}-{md}'] = dataset

horizontal square
horizontal circle
horizontal triangle


In [188]:
# Create test environments and datasets
tam_test_datasets = defaultdict()
motion_test_datasets = defaultdict()
tam_types = ["tam", "outline"]
motion_types = ["cnt", "track"]

for bs in box_shapes:
    for tt in tam_types:
        for md in motion_dirs:
            print(bs, tt, md)
            if tt == "tam":
                _path = STIM_DIR.parent / "TAM_basic"
            else:
                _path = STIM_DIR.parent / "TAM_outline"

            stimuli = load_tam_from_image(img_size, stim_type=tt, stim_path=_path)
            env = TAMTask(
                box_shape=bs,
                stimuli=stimuli,
                stim_type=tt,
                stim_ori=md,
                dt=env_params["dt"],
                sigma=0.01,
                img_size=img_size
            )
            env.reset(no_step=True)

            # Generate dataset object
            dataset = ngym.Dataset(
                env,
                batch_size=batch_size,
                seq_len=seq_len
            )

            # Add dataset to dictionary
            tam_test_datasets[f'{bs}-{tt}-{md}'] = env
            tam_test_datasets[f'{bs}-{tt}-{md}'] = dataset
#
# for bs in box_shapes:
#     for mt in motion_types:
#         for md in motion_dirs:
#
#             print(bs, mt, md)
#
#             stimuli = load_motion_from_image(img_size, stim_path=STIM_DIR.parent / "motion_task")
#             env = MotionTask(
#                 dt=env_params["dt"],
#                 box_shape=bs,
#                 motion_type=mt,
#                 stim_ori=md,
#                 stimuli=stimuli,
#                 sigma=0,
#                 img_size=img_size
#             )
#             env.reset(no_step=True)
#
#             # Generate dataset object
#             dataset = ngym.Dataset(
#                 env,
#                 batch_size=batch_size,
#                 seq_len=seq_len
#             )
#
#             # Add dataset to dictionary
#             motion_test_datasets[f'{bs}-{mt}-{md}'] = env
#             motion_test_datasets[f'{bs}-{mt}-{md}'] = dataset

square tam horizontal
square outline horizontal
circle tam horizontal
circle outline horizontal
triangle tam horizontal
triangle outline horizontal


In [189]:
# Look at the shapes of the datasets
inputs, target = training_datasets["square-horizontal"]()
print('Input has shape (SeqLen, Batch, C, W, H) =', inputs.shape)
print('Target has shape (SeqLen, Batch) =', target.shape)
print('Targets in the first sequence:')
print(target[:, 0])

Input has shape (SeqLen, Batch, C, W, H) = (100, 128, 32, 32)
Target has shape (SeqLen, Batch) = (100, 128)
Targets in the first sequence:
[0 0 0 0 0 0 0 3 3 3 0 0 0 0 0 0 0 2 2 2 0 0 0 0 0 0 0 3 3 3 0 0 0 0 0 0 0
 1 1 1 0 0 0 0 0 0 0 2 2 2 0 0 0 0 0 0 0 2 2 2 0 0 0 0 0 0 0 3 3 3 0 0 0 0
 0 0 0 1 1 1 0 0 0 0 0 0 0 2 2 2 0 0 0 0 0 0 0 2 2 2]


In [191]:
# Load parameters
device = torch.device(net_params["device"])
# device = torch.device('cpu')

# Create network to train on the normal stimuli
net = TAMNet(
    dataset=training_datasets["square-horizontal"],
    hidden_size=256,
    output_size=training_datasets["square-horizontal"].env.action_space.n,
    learning_rate=0.01,
    device=device
)

In [192]:
# Train network
train_loss, train_acc = net.train_rnn(
    datasets=[
        training_datasets["square-horizontal"],
        tam_test_datasets["square-tam-horizontal"],
        # training_datasets["triangle-horizontal"],
        # training_datasets["circle-horizontal"],
    ],
    # n_epochs=net_params["n_epochs"]
    n_epochs=200
)

Training started...
Step 100, Loss 0.3731, Time 89.6s
Step 200, Loss 0.0008, Time 181.1s


In [143]:
net.RNN.rnn.train()
net.RNN.train()

RNNNet(
  (rnn): CTRNN(
    (input2h): Linear(in_features=256, out_features=256, bias=True)
    (h2h): Linear(in_features=256, out_features=256, bias=True)
  )
  (fc): Linear(in_features=256, out_features=6, bias=True)
)

In [193]:
net.RNN.rnn.eval()
net.RNN.eval()

RNNNet(
  (rnn): CTRNN(
    (input2h): Linear(in_features=1024, out_features=256, bias=True)
    (h2h): Linear(in_features=256, out_features=256, bias=True)
  )
  (fc): Linear(in_features=256, out_features=4, bias=True)
)

In [194]:
# Test on other environments
test_accuracies = defaultdict()
test_activities = defaultdict()
infos = defaultdict()
for et, dataset in tam_test_datasets.items():
    info, activity, test_acc = net.test_rnn(100, dataset)
    infos[et] = info
    test_accuracies[et] = test_acc
    test_activities[et] = activity
    print(f'Accuracy on {et} = {test_acc}')

Accuracy on square-tam-horizontal = 1.0
Accuracy on square-outline-horizontal = 0.73
Accuracy on circle-tam-horizontal = 0.8
Accuracy on circle-outline-horizontal = 0.32
Accuracy on triangle-tam-horizontal = 0.8
Accuracy on triangle-outline-horizontal = 0.28


In [195]:
# Test on other environments
test_accuracies = defaultdict()
test_activities = defaultdict()
infos = defaultdict()
for et, dataset in training_datasets.items():
    info, activity, test_acc = net.test_rnn(100, dataset)
    infos[et] = info
    test_accuracies[et] = test_acc
    test_activities[et] = activity
    print(f'Accuracy on {et} = {test_acc}')

Accuracy on square-horizontal = 1.0
Accuracy on circle-horizontal = 0.99
Accuracy on triangle-horizontal = 0.54


In [196]:
# save network
torch.save(net.RNN.state_dict(), MODEL_DIR / f'learned_horiz_both.pt')